# Keyword & Theme Deep-Dive Analysis
## From Analytics to Action — DTU Spring 2026

**Case company:** Publikum (formerly Will & Agency)  
**Dataset:** 2,000 European movies from IMDb  
**Purpose:** Comprehensive analysis of the `keywords` column — exploring thematic patterns, correlations with ratings/popularity/genre/country/time, network structures, clustering, and predictive power.

---

### Why keywords?

Keywords are user-contributed thematic tags on IMDb. Unlike genres (which are broad industry labels), keywords capture **specific narrative elements, visual content, settings, relationships, and production context**. With ~56,000 unique keywords across our 2,000 films, this is the richest descriptive dimension in the dataset — and the closest analog to the thematic analysis that Publikum's PlotBounce tool performs at scale.

> **Datafication caveat:** Keywords are user-contributed and non-standardized. "world war ii" and "ww2" may coexist as separate tags. Our analysis inherits these inconsistencies.

---

### Table of Contents

1. [Setup & Imports](#1-setup)
2. [Load Data & Prepare Keywords](#2-load)
3. [Keyword Frequency & Distribution](#3-frequency)
4. [Keywords vs. Ratings](#4-ratings)
5. [Keywords vs. Popularity](#5-popularity)
6. [Keywords vs. Genre](#6-genre)
7. [Keywords vs. Country](#7-country)
8. [Keywords vs. Time](#8-time)
9. [Keyword Co-occurrence Network](#9-network)
10. [Keyword Clustering / Thematic Grouping](#10-clustering)
11. [Keyword Category Analysis](#11-categories)
12. [Keyword Rarity & Uniqueness](#12-rarity)
13. [Keywords & Runtime](#13-runtime)
14. [Keywords & Co-productions](#14-coproductions)
15. [Predictive: Can Keywords Predict Rating Tier?](#15-predictive)
16. [Summary & Key Takeaways](#16-summary)

<a id="1-setup"></a>
## 1. Setup & Imports

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter
from itertools import combinations
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# scikit-learn for TF-IDF, clustering, and prediction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix

# Change to project root
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

os.makedirs('reports/figures', exist_ok=True)

from src.visualize import set_style, save_figure
set_style()

print(f"Working directory: {os.getcwd()}")
print("All libraries loaded successfully!")

<a id="2-load"></a>
## 2. Load Data & Prepare Keywords

In [ ]:
# Load the dataset
df = pd.read_excel('03-data/European_data_2000.xlsx')
print(f"Dataset shape: {df.shape[0]:,} movies x {df.shape[1]} columns")

# Helper function (same as general_analysis.ipynb)
def split_field(series, sep=', '):
    """Split a comma-separated string column into a flat Series of individual values."""
    return series.dropna().str.split(sep).explode().str.strip()

# Create derived columns
df['keyword_list'] = df['keywords'].str.split(', ')
df['keyword_count'] = df['keyword_list'].apply(lambda x: len(x) if isinstance(x, list) else 0)
df['genre_count'] = df['genres'].str.split(', ').apply(lambda x: len(x) if isinstance(x, list) else 0)
df['country_count'] = df['allCountries'].str.split(', ').apply(lambda x: len(x) if isinstance(x, list) else 0)
df['is_coproduction'] = df['country_count'] > 1
df['decade'] = (df['releaseYear'] // 10) * 10

# Build global keyword counts
all_keywords = split_field(df['keywords'])
keyword_counts = all_keywords.value_counts()

print(f"\n=== Keyword Summary ===")
print(f"Total keyword tags across all films: {len(all_keywords):,}")
print(f"Unique keywords: {len(keyword_counts):,}")
print(f"Keywords per film — mean: {df['keyword_count'].mean():.1f}, median: {df['keyword_count'].median():.0f}, max: {df['keyword_count'].max()}")

<a id="3-frequency"></a>
## 3. Keyword Frequency & Distribution

Before analyzing relationships, we need to understand the **shape** of the keyword data. How concentrated or dispersed are keywords? Does the distribution follow Zipf's law (a power-law pattern common in language data)?

In [ ]:
# Top 40 keywords
fig, ax = plt.subplots(figsize=(10, 10))
keyword_counts.head(40).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Films')
ax.set_title('Top 40 Keywords Across European Films (n=2,000)')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'kw_top40')
plt.show()

In [ ]:
# Zipf's law: log-log plot of rank vs. frequency
ranks = np.arange(1, len(keyword_counts) + 1)
freqs = keyword_counts.values

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(ranks, freqs, s=2, alpha=0.5, color='steelblue')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Keyword Rank (log)')
ax.set_ylabel('Frequency (log)')
ax.set_title("Zipf's Law: Keyword Frequency vs. Rank\n(A straight line indicates a power-law distribution)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, 'kw_zipf_distribution')
plt.show()

In [ ]:
# Keywords per film distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['keyword_count'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['keyword_count'].mean(), color='red', linestyle='--',
           label=f'Mean: {df["keyword_count"].mean():.0f}')
ax.axvline(df['keyword_count'].median(), color='orange', linestyle='--',
           label=f'Median: {df["keyword_count"].median():.0f}')
ax.set_xlabel('Number of Keywords per Film')
ax.set_ylabel('Number of Films')
ax.set_title('Distribution of Keyword Count per Film')
ax.legend()
plt.tight_layout()
save_figure(fig, 'kw_count_per_film')
plt.show()

# Long-tail statistics
hapax = (keyword_counts == 1).sum()
rare = (keyword_counts <= 5).sum()
common = (keyword_counts >= len(df) * 0.5).sum()
print(f"\nKeywords appearing only once (hapax legomena): {hapax:,} ({hapax/len(keyword_counts)*100:.1f}%)")
print(f"Keywords appearing ≤5 times: {rare:,} ({rare/len(keyword_counts)*100:.1f}%)")
print(f"Keywords in ≥50% of films: {common}")

<a id="4-ratings"></a>
## 4. Keywords vs. Ratings

Do certain keywords correlate with higher or lower IMDb ratings? This is one of the most actionable analyses — it reveals which **thematic elements** are associated with critical acclaim or poor reception.

In [ ]:
# Compute mean rating for each keyword (min 20 films to be meaningful)
MIN_FILMS = 20

kw_rating_stats = []
for kw, count in keyword_counts.items():
    if count >= MIN_FILMS:
        mask = df['keywords'].str.contains(rf'(?:^|, ){kw}(?:,|$)', na=False, regex=True)
        if mask.sum() >= MIN_FILMS:
            ratings = df.loc[mask, 'imdbRating']
            kw_rating_stats.append({
                'keyword': kw,
                'mean_rating': ratings.mean(),
                'median_rating': ratings.median(),
                'std_rating': ratings.std(),
                'count': mask.sum()
            })

kw_ratings = pd.DataFrame(kw_rating_stats).sort_values('mean_rating', ascending=False)
print(f"Keywords with {MIN_FILMS}+ films: {len(kw_ratings)}")

In [ ]:
# Top 20 highest-rated and bottom 20 lowest-rated keywords
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top20 = kw_ratings.head(20)
axes[0].barh(range(len(top20)), top20['mean_rating'], color='#2ecc71')
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels([f"{r['keyword']} (n={r['count']})" for _, r in top20.iterrows()])
axes[0].set_xlabel('Mean IMDb Rating')
axes[0].set_title('Top 20 Highest-Rated Keywords')
axes[0].invert_yaxis()
axes[0].set_xlim(left=df['imdbRating'].mean() - 1)

bottom20 = kw_ratings.tail(20)
axes[1].barh(range(len(bottom20)), bottom20['mean_rating'], color='#e74c3c')
axes[1].set_yticks(range(len(bottom20)))
axes[1].set_yticklabels([f"{r['keyword']} (n={r['count']})" for _, r in bottom20.iterrows()])
axes[1].set_xlabel('Mean IMDb Rating')
axes[1].set_title('Bottom 20 Lowest-Rated Keywords')
axes[1].invert_yaxis()

plt.tight_layout()
save_figure(fig, 'kw_highest_lowest_rated')
plt.show()

In [ ]:
# Scatter: keyword frequency vs. mean rating
# Common keywords should regress to the mean; rare keywords can be extreme
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(kw_ratings['count'], kw_ratings['mean_rating'],
                     s=15, alpha=0.5, c='steelblue')
ax.axhline(df['imdbRating'].mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Overall mean: {df["imdbRating"].mean():.2f}')
ax.set_xlabel('Keyword Frequency (number of films)')
ax.set_ylabel('Mean IMDb Rating')
ax.set_title('Keyword Frequency vs. Mean Rating\n(Common keywords converge to the mean)')
ax.legend()

# Label outliers
for _, row in kw_ratings.head(5).iterrows():
    ax.annotate(row['keyword'], (row['count'], row['mean_rating']),
                fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')
for _, row in kw_ratings.tail(5).iterrows():
    ax.annotate(row['keyword'], (row['count'], row['mean_rating']),
                fontsize=7, alpha=0.8, xytext=(5, -10), textcoords='offset points')

plt.tight_layout()
save_figure(fig, 'kw_frequency_vs_rating')
plt.show()

In [ ]:
# Statistical significance: t-tests for top 50 keywords
# Do films with keyword X have a significantly different rating than films without?
overall_mean = df['imdbRating'].mean()
n_tests = 50
alpha = 0.05 / n_tests  # Bonferroni correction

sig_results = []
for _, row in kw_ratings.head(n_tests).iterrows():
    kw = row['keyword']
    mask = df['keywords'].str.contains(rf'(?:^|, ){kw}(?:,|$)', na=False, regex=True)
    with_kw = df.loc[mask, 'imdbRating']
    without_kw = df.loc[~mask, 'imdbRating']
    t_stat, p_val = stats.ttest_ind(with_kw, without_kw)
    sig_results.append({
        'keyword': kw,
        'mean_with': with_kw.mean(),
        'mean_without': without_kw.mean(),
        'diff': with_kw.mean() - without_kw.mean(),
        'p_value': p_val,
        'significant': p_val < alpha
    })

sig_df = pd.DataFrame(sig_results)
sig_only = sig_df[sig_df['significant']].sort_values('diff', ascending=False)
print(f"=== Statistically Significant Keywords (Bonferroni α={alpha:.4f}) ===")
print(f"{len(sig_only)} of {n_tests} keywords are significant\n")
if len(sig_only) > 0:
    print(sig_only[['keyword', 'mean_with', 'mean_without', 'diff', 'p_value']]
          .to_string(index=False, float_format='%.3f'))

In [ ]:
# Box plots for 10 interesting keywords (5 highest, 5 lowest rated)
selected_high = kw_ratings.head(5)['keyword'].tolist()
selected_low = kw_ratings.tail(5)['keyword'].tolist()
selected_kws = selected_high + selected_low

box_data = []
for kw in selected_kws:
    mask = df['keywords'].str.contains(rf'(?:^|, ){kw}(?:,|$)', na=False, regex=True)
    for rating in df.loc[mask, 'imdbRating']:
        box_data.append({'keyword': kw, 'rating': rating})

box_df = pd.DataFrame(box_data)

fig, ax = plt.subplots(figsize=(14, 6))
order = selected_kws
colors = ['#2ecc71'] * 5 + ['#e74c3c'] * 5
sns.boxplot(data=box_df, x='keyword', y='rating', order=order, ax=ax, palette=colors)
ax.axhline(df['imdbRating'].mean(), color='black', linestyle='--', alpha=0.5,
           label=f'Overall mean: {df["imdbRating"].mean():.2f}')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_ylabel('IMDb Rating')
ax.set_title('Rating Distribution: Highest vs. Lowest-Rated Keywords')
ax.legend()
plt.tight_layout()
save_figure(fig, 'kw_rating_boxplots')
plt.show()

<a id="5-popularity"></a>
## 5. Keywords vs. Popularity (Number of Votes)

Ratings measure perceived quality; vote counts measure **visibility and engagement**. A keyword might appear in highly-rated but obscure films, or in mediocre but widely-seen ones. This section separates "mainstream" from "arthouse" keyword signatures.

In [ ]:
# Compute median votes per keyword
kw_vote_stats = []
for kw, count in keyword_counts.items():
    if count >= MIN_FILMS:
        mask = df['keywords'].str.contains(rf'(?:^|, ){kw}(?:,|$)', na=False, regex=True)
        if mask.sum() >= MIN_FILMS:
            votes = df.loc[mask, 'numberOfVotes']
            kw_vote_stats.append({
                'keyword': kw,
                'median_votes': votes.median(),
                'mean_votes': votes.mean(),
                'count': mask.sum()
            })

kw_votes = pd.DataFrame(kw_vote_stats).sort_values('median_votes', ascending=False)

# Top 20 most popular keywords by median votes
fig, ax = plt.subplots(figsize=(10, 8))
top_pop = kw_votes.head(20)
ax.barh(range(len(top_pop)), top_pop['median_votes'], color='coral')
ax.set_yticks(range(len(top_pop)))
ax.set_yticklabels([f"{r['keyword']} (n={r['count']})" for _, r in top_pop.iterrows()])
ax.set_xlabel('Median Number of Votes')
ax.set_title('Top 20 Keywords by Median Popularity (Votes)')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'kw_most_popular')
plt.show()

In [ ]:
# Popular vs. Niche films: which keywords distinguish them?
q75 = df['numberOfVotes'].quantile(0.75)
q25 = df['numberOfVotes'].quantile(0.25)

popular_films = df[df['numberOfVotes'] >= q75]
niche_films = df[df['numberOfVotes'] <= q25]

pop_kw = split_field(popular_films['keywords']).value_counts()
niche_kw = split_field(niche_films['keywords']).value_counts()

# Compute popularity ratio for keywords appearing in both groups
pop_pct = pop_kw / len(popular_films)
niche_pct = niche_kw / len(niche_films)

# Combine and compute ratio (only keywords in both groups with min count)
common_kws = set(pop_kw[pop_kw >= 10].index) & set(niche_kw[niche_kw >= 10].index)
ratios = []
for kw in common_kws:
    ratio = pop_pct.get(kw, 0) / max(niche_pct.get(kw, 0.001), 0.001)
    ratios.append({'keyword': kw, 'popularity_ratio': ratio,
                   'pop_pct': pop_pct.get(kw, 0) * 100,
                   'niche_pct': niche_pct.get(kw, 0) * 100})

ratio_df = pd.DataFrame(ratios).sort_values('popularity_ratio', ascending=False)

# Diverging bar chart: most mainstream vs most arthouse keywords
top_mainstream = ratio_df.head(15)
top_arthouse = ratio_df.tail(15).iloc[::-1]
combined = pd.concat([top_mainstream, top_arthouse])

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['coral' if r > 1 else 'steelblue' for r in combined['popularity_ratio']]
ax.barh(range(len(combined)), np.log2(combined['popularity_ratio']), color=colors)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(combined['keyword'])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('log₂(Popularity Ratio)  ←Arthouse | Mainstream→')
ax.set_title('Mainstream vs. Arthouse Keywords\n(Ratio of prevalence in top-25% vs bottom-25% voted films)')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'kw_popularity_ratio')
plt.show()

<a id="6-genre"></a>
## 6. Keywords vs. Genre

Genres are broad labels; keywords are specific. How much do keywords **overlap with or extend beyond** genre information? This section explores distinctive keyword profiles per genre and tests whether keywords can predict genre membership.

In [ ]:
# Heatmap: top 8 genres × top 20 keywords (normalized by genre size)
top_8_genres = split_field(df['genres']).value_counts().head(8).index.tolist()
top_20_kws = keyword_counts.head(20).index.tolist()

genre_kw_matrix = pd.DataFrame(0.0, index=top_8_genres, columns=top_20_kws)

for genre in top_8_genres:
    genre_mask = df['genres'].str.contains(genre, na=False)
    n_genre = genre_mask.sum()
    genre_kws = split_field(df.loc[genre_mask, 'keywords'])
    for kw in top_20_kws:
        genre_kw_matrix.loc[genre, kw] = (genre_kws == kw).sum() / n_genre * 100

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(genre_kw_matrix, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Keyword Prevalence by Genre (% of films in genre containing keyword)')
ax.set_ylabel('Genre')
ax.set_xlabel('Keyword')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure(fig, 'kw_genre_heatmap')
plt.show()

In [ ]:
# TF-IDF: find keywords that are DISTINCTIVE to each genre
# Treat each genre as a "document" whose text is all keywords from films in that genre

genre_docs = []
for genre in top_8_genres:
    genre_mask = df['genres'].str.contains(genre, na=False)
    all_kws = ' '.join(df.loc[genre_mask, 'keywords'].dropna().str.replace(', ', ' '))
    genre_docs.append(all_kws)

tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(genre_docs)
feature_names = tfidf.get_feature_names_out()

print("=== Most Distinctive Keywords per Genre (TF-IDF) ===\n")
for i, genre in enumerate(top_8_genres):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[-10:][::-1]
    top_words = [(feature_names[j], scores[j]) for j in top_idx]
    print(f"{genre}:")
    for word, score in top_words:
        print(f"  {word}: {score:.3f}")
    print()

In [ ]:
# Genre prediction from keywords
# Can we predict a film's genre just from its keywords?

# Build binary keyword matrix (only keywords in 10+ films for tractability)
frequent_kws = keyword_counts[keyword_counts >= 10].index.tolist()
mlb = MultiLabelBinarizer(classes=frequent_kws)
kw_matrix = mlb.fit_transform(df['keyword_list'].apply(
    lambda x: [k for k in x if k in frequent_kws] if isinstance(x, list) else []))

print(f"Keyword matrix shape: {kw_matrix.shape} (films × keywords)")

# Predict each genre independently
genre_f1_scores = {}
for genre in top_8_genres:
    y = df['genres'].str.contains(genre, na=False).astype(int)
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, kw_matrix, y, cv=5, scoring='f1')
    genre_f1_scores[genre] = scores.mean()
    print(f"{genre}: F1 = {scores.mean():.3f} (±{scores.std():.3f})")

fig, ax = plt.subplots(figsize=(10, 5))
genres_sorted = sorted(genre_f1_scores.items(), key=lambda x: x[1], reverse=True)
ax.bar([g[0] for g in genres_sorted], [g[1] for g in genres_sorted], color='steelblue')
ax.set_ylabel('F1 Score (5-fold CV)')
ax.set_title('Can Keywords Predict Genre? (RandomForest F1 Score)')
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure(fig, 'kw_genre_prediction_f1')
plt.show()

<a id="7-country"></a>
## 7. Keywords vs. Country

Do different national cinemas have distinct thematic profiles? This section explores whether keyword patterns reveal cultural differences in European filmmaking — and highlights what makes Nordic cinema distinctive.

In [ ]:
# Heatmap: top 10 countries × top 20 keywords
top_10_countries = df['mainCountry'].value_counts().head(10).index.tolist()

country_kw_matrix = pd.DataFrame(0.0, index=top_10_countries, columns=top_20_kws)

for country in top_10_countries:
    country_mask = df['mainCountry'] == country
    n_country = country_mask.sum()
    country_kws = split_field(df.loc[country_mask, 'keywords'])
    for kw in top_20_kws:
        country_kw_matrix.loc[country, kw] = (country_kws == kw).sum() / n_country * 100

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(country_kw_matrix, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Keyword Prevalence by Country (% of country\'s films containing keyword)')
ax.set_ylabel('Country')
ax.set_xlabel('Keyword')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure(fig, 'kw_country_heatmap')
plt.show()

In [ ]:
# TF-IDF distinctive keywords per country
country_docs = []
for country in top_10_countries:
    country_mask = df['mainCountry'] == country
    all_kws = ' '.join(df.loc[country_mask, 'keywords'].dropna().str.replace(', ', ' '))
    country_docs.append(all_kws)

tfidf_country = TfidfVectorizer(max_features=5000)
tfidf_c_matrix = tfidf_country.fit_transform(country_docs)
c_feature_names = tfidf_country.get_feature_names_out()

print("=== Most Distinctive Keywords per Country (TF-IDF) ===\n")
for i, country in enumerate(top_10_countries):
    scores = tfidf_c_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[-10:][::-1]
    top_words = [(c_feature_names[j], scores[j]) for j in top_idx]
    print(f"{country}:")
    for word, score in top_words:
        print(f"  {word}: {score:.3f}")
    print()

In [ ]:
# Nordic vs. Rest of Europe: diverging keyword comparison
nordic_countries = ['DK', 'SE', 'NO', 'FI', 'IS']
nordic_mask = df['mainCountry'].isin(nordic_countries)

nordic_kw = split_field(df.loc[nordic_mask, 'keywords']).value_counts()
rest_kw = split_field(df.loc[~nordic_mask, 'keywords']).value_counts()

nordic_pct = nordic_kw / nordic_mask.sum()
rest_pct = rest_kw / (~nordic_mask).sum()

# Keywords in both groups with min count
common_n = set(nordic_kw[nordic_kw >= 5].index) & set(rest_kw[rest_kw >= 20].index)
nordic_ratios = []
for kw in common_n:
    ratio = nordic_pct.get(kw, 0) / max(rest_pct.get(kw, 0.001), 0.001)
    nordic_ratios.append({'keyword': kw, 'ratio': ratio})

nr_df = pd.DataFrame(nordic_ratios).sort_values('ratio', ascending=False)

top_nordic = nr_df.head(15)
top_rest = nr_df.tail(15).iloc[::-1]
combined_nr = pd.concat([top_nordic, top_rest])

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#1a5276' if r > 1 else '#c0392b' for r in combined_nr['ratio']]
ax.barh(range(len(combined_nr)), np.log2(combined_nr['ratio']), color=colors)
ax.set_yticks(range(len(combined_nr)))
ax.set_yticklabels(combined_nr['keyword'])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('log₂(Nordic/Rest ratio)  ←More in Rest | More in Nordic→')
ax.set_title('Keywords: Nordic Cinema vs. Rest of Europe')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'kw_nordic_vs_rest')
plt.show()

print(f"\nNordic films in dataset: {nordic_mask.sum()}")
print(f"Rest of Europe: {(~nordic_mask).sum()}")

<a id="8-time"></a>
## 8. Keywords vs. Time (Decade)

Has the thematic landscape of European cinema shifted over time? This section identifies trending, emerging, and declining keyword themes across decades.

In [ ]:
# Heatmap: decades × top 20 keywords
# Filter to decades with enough films
decade_counts = df['decade'].value_counts()
valid_decades = sorted(decade_counts[decade_counts >= 30].index)

decade_kw_matrix = pd.DataFrame(0.0, index=valid_decades, columns=top_20_kws)

for decade in valid_decades:
    dec_mask = df['decade'] == decade
    n_dec = dec_mask.sum()
    dec_kws = split_field(df.loc[dec_mask, 'keywords'])
    for kw in top_20_kws:
        decade_kw_matrix.loc[decade, kw] = (dec_kws == kw).sum() / n_dec * 100

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(decade_kw_matrix, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Keyword Prevalence by Decade (% of films in decade)')
ax.set_ylabel('Decade')
ax.set_xlabel('Keyword')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure(fig, 'kw_decade_heatmap')
plt.show()

In [ ]:
# Trending keywords: compute slope of prevalence over time
# Use linear regression on (decade, prevalence %) for keywords with enough data

trend_results = []
for kw in keyword_counts[keyword_counts >= 30].index[:200]:  # Check top 200
    prevalences = []
    for decade in valid_decades:
        dec_mask = df['decade'] == decade
        n_dec = dec_mask.sum()
        kw_in_dec = df.loc[dec_mask, 'keywords'].str.contains(
            rf'(?:^|, ){kw}(?:,|$)', na=False, regex=True).sum()
        prevalences.append(kw_in_dec / n_dec * 100)
    
    if len(valid_decades) >= 3:
        slope, _, r_value, _, _ = stats.linregress(range(len(valid_decades)), prevalences)
        trend_results.append({
            'keyword': kw, 'slope': slope, 'r_squared': r_value**2,
            'prevalences': prevalences
        })

trend_df = pd.DataFrame(trend_results).sort_values('slope', ascending=False)

# Plot top 5 rising and top 5 declining
rising = trend_df.head(5)
declining = trend_df.tail(5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for _, row in rising.iterrows():
    axes[0].plot(valid_decades, row['prevalences'], 'o-', label=row['keyword'])
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('% of Films')
axes[0].set_title('Top 5 Rising Keywords')
axes[0].legend(fontsize=8)

for _, row in declining.iterrows():
    axes[1].plot(valid_decades, row['prevalences'], 'o-', label=row['keyword'])
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('% of Films')
axes[1].set_title('Top 5 Declining Keywords')
axes[1].legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'kw_trending_over_time')
plt.show()

In [ ]:
# Emerging keywords: appear primarily in post-2010 films
post_2010 = df[df['releaseYear'] >= 2010]
pre_2000 = df[df['releaseYear'] < 2000]

post_kw = split_field(post_2010['keywords']).value_counts()
pre_kw = split_field(pre_2000['keywords']).value_counts()

# Keywords common post-2010 but rare/absent pre-2000
emerging = []
for kw in post_kw[post_kw >= 15].index:
    pre_count = pre_kw.get(kw, 0)
    post_pct = post_kw[kw] / len(post_2010) * 100
    pre_pct = pre_count / max(len(pre_2000), 1) * 100
    if pre_pct < 1 and post_pct > 2:
        emerging.append({'keyword': kw, 'post_2010_%': round(post_pct, 1),
                        'pre_2000_%': round(pre_pct, 1)})

emerging_df = pd.DataFrame(emerging).sort_values('post_2010_%', ascending=False).head(20)
print("=== Emerging Keywords (common post-2010, rare pre-2000) ===")
print(emerging_df.to_string(index=False))

# Declining keywords: common pre-2000 but rare post-2010
declining_kws = []
for kw in pre_kw[pre_kw >= 10].index:
    post_count = post_kw.get(kw, 0)
    pre_pct = pre_kw[kw] / len(pre_2000) * 100
    post_pct = post_count / max(len(post_2010), 1) * 100
    if post_pct < 1 and pre_pct > 3:
        declining_kws.append({'keyword': kw, 'pre_2000_%': round(pre_pct, 1),
                             'post_2010_%': round(post_pct, 1)})

declining_kws_df = pd.DataFrame(declining_kws).sort_values('pre_2000_%', ascending=False).head(20)
print("\n=== Declining Keywords (common pre-2000, rare post-2010) ===")
print(declining_kws_df.to_string(index=False))

<a id="9-network"></a>
## 9. Keyword Co-occurrence Network

Following the **Visual Network Analysis** framework (Venturini et al., 2021), we build a network where keywords are nodes and edges represent co-occurrence on the same film. This reveals thematic **clusters** and **bridge themes** connecting different content domains.

In [ ]:
# Build co-occurrence network for top 100 keywords
top_100_kws = keyword_counts.head(100).index.tolist()

G = nx.Graph()
for kw in top_100_kws:
    G.add_node(kw, frequency=keyword_counts[kw])

# Count co-occurrences
cooc_counts = Counter()
for kw_list in df['keyword_list'].dropna():
    filtered = [k for k in kw_list if k in top_100_kws]
    for k1, k2 in combinations(filtered, 2):
        pair = tuple(sorted([k1, k2]))
        cooc_counts[pair] += 1

# Add edges with weight threshold
min_cooc = 50
for (k1, k2), weight in cooc_counts.items():
    if weight >= min_cooc:
        G.add_edge(k1, k2, weight=weight)

# Remove isolated nodes
isolated = [n for n in G.nodes() if G.degree(n) == 0]
G.remove_nodes_from(isolated)

print(f"Network: {G.number_of_nodes()} keywords, {G.number_of_edges()} co-occurrence links")
print(f"(Threshold: {min_cooc}+ co-occurrences, removed {len(isolated)} isolated nodes)")

In [ ]:
# Community detection
communities = nx.community.louvain_communities(G, weight='weight', seed=42)

# Assign community labels to nodes
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

print(f"Detected {len(communities)} communities:\n")
for i, comm in enumerate(communities):
    members = sorted(comm, key=lambda x: keyword_counts[x], reverse=True)
    print(f"Community {i} ({len(comm)} keywords): {', '.join(members[:8])}{'...' if len(members) > 8 else ''}")

In [ ]:
# Network visualization
fig, ax = plt.subplots(figsize=(18, 14))

pos = nx.spring_layout(G, k=2.5, iterations=80, seed=42, weight='weight')

# Node properties
node_sizes = [G.nodes[n].get('frequency', 50) * 0.8 for n in G.nodes()]
node_colors = [community_map.get(n, 0) for n in G.nodes()]

# Edge properties
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights)
edge_widths = [w / max_w * 4 + 0.3 for w in edge_weights]

nx.draw_networkx_edges(G, pos, alpha=0.15, width=edge_widths, edge_color='gray', ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                       cmap=plt.cm.Set3, alpha=0.8, ax=ax)

# Label top nodes by degree
top_by_degree = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:40]
labels = {n: n for n, _ in top_by_degree}
nx.draw_networkx_labels(G, pos, labels, font_size=7, font_weight='bold', ax=ax)

ax.set_title('Keyword Co-occurrence Network\n'
             'Node size = frequency, Color = community, Edge thickness = co-occurrence strength',
             fontsize=14)
ax.axis('off')
plt.tight_layout()
save_figure(fig, 'kw_cooccurrence_network')
plt.show()

In [ ]:
# Centrality metrics
print("=== Top 10 Keywords by Degree Centrality ===")
print("(Most connected — co-occur with many other keywords)\n")
degree_c = nx.degree_centrality(G)
for node, score in sorted(degree_c.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node}: {score:.3f}")

print("\n=== Top 10 Keywords by Betweenness Centrality ===")
print("(Bridge keywords connecting different thematic clusters)\n")
between_c = nx.betweenness_centrality(G, weight='weight')
for node, score in sorted(between_c.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node}: {score:.3f}")

<a id="10-clustering"></a>
## 10. Keyword Clustering / Thematic Grouping

Can we automatically discover thematic clusters among keywords using unsupervised learning? We build keyword vectors from co-occurrence patterns and apply KMeans clustering.

In [ ]:
# Build keyword co-occurrence vectors for top 200 keywords
top_200_kws = keyword_counts.head(200).index.tolist()

# Create a film × keyword binary matrix
kw_to_idx = {kw: i for i, kw in enumerate(top_200_kws)}
film_kw_matrix = np.zeros((len(df), len(top_200_kws)))

for i, kw_list in enumerate(df['keyword_list']):
    if isinstance(kw_list, list):
        for kw in kw_list:
            if kw in kw_to_idx:
                film_kw_matrix[i, kw_to_idx[kw]] = 1

# Compute keyword-keyword co-occurrence matrix (Jaccard similarity)
# Jaccard(A, B) = |A ∩ B| / |A ∪ B|
cooc_matrix = film_kw_matrix.T @ film_kw_matrix  # co-occurrence counts
row_sums = film_kw_matrix.sum(axis=0)

jaccard_matrix = np.zeros((len(top_200_kws), len(top_200_kws)))
for i in range(len(top_200_kws)):
    for j in range(len(top_200_kws)):
        union = row_sums[i] + row_sums[j] - cooc_matrix[i, j]
        if union > 0:
            jaccard_matrix[i, j] = cooc_matrix[i, j] / union

print(f"Jaccard similarity matrix: {jaccard_matrix.shape}")

In [ ]:
# Elbow method to choose k
inertias = []
K_range = range(4, 16)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(jaccard_matrix)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_range), inertias, 'o-', color='steelblue')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method for Keyword Clustering')
plt.tight_layout()
save_figure(fig, 'kw_cluster_elbow')
plt.show()

In [ ]:
# Apply KMeans with chosen k (8 clusters as a reasonable default)
K = 8
km = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = km.fit_predict(jaccard_matrix)

# Display clusters
print(f"=== {K} Keyword Clusters ===\n")
cluster_data = []
for c in range(K):
    members = [top_200_kws[i] for i in range(len(top_200_kws)) if cluster_labels[i] == c]
    members_sorted = sorted(members, key=lambda x: keyword_counts[x], reverse=True)
    print(f"Cluster {c} ({len(members)} keywords):")
    print(f"  {', '.join(members_sorted[:12])}{'...' if len(members) > 12 else ''}")
    print()
    
    # Compute cluster profile
    cluster_mask = df['keywords'].apply(
        lambda x: any(kw in str(x) for kw in members_sorted[:5]) if pd.notna(x) else False)
    cluster_data.append({
        'cluster': c,
        'size': len(members),
        'top_keywords': ', '.join(members_sorted[:5]),
        'mean_rating': df.loc[cluster_mask, 'imdbRating'].mean(),
        'mean_votes': df.loc[cluster_mask, 'numberOfVotes'].mean()
    })

In [ ]:
# PCA visualization of keyword clusters
pca = PCA(n_components=2)
kw_2d = pca.fit_transform(jaccard_matrix)

fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(kw_2d[:, 0], kw_2d[:, 1], c=cluster_labels,
                     cmap='Set3', s=50, alpha=0.7)

# Label a selection of keywords
for i, kw in enumerate(top_200_kws):
    if keyword_counts[kw] >= 200 or i < 30:  # Label frequent ones
        ax.annotate(kw, (kw_2d[i, 0], kw_2d[i, 1]),
                    fontsize=6, alpha=0.8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('Keyword Space (PCA of Jaccard Co-occurrence)\nColored by KMeans Cluster')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
save_figure(fig, 'kw_cluster_pca')
plt.show()

In [ ]:
# Cluster profiles: mean rating and votes
cluster_profile_df = pd.DataFrame(cluster_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cluster_profile_df['cluster'], cluster_profile_df['mean_rating'], color='steelblue')
axes[0].axhline(df['imdbRating'].mean(), color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Mean IMDb Rating')
axes[0].set_title('Mean Rating by Keyword Cluster')

axes[1].bar(cluster_profile_df['cluster'], cluster_profile_df['mean_votes'], color='coral')
axes[1].axhline(df['numberOfVotes'].mean(), color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Mean Votes')
axes[1].set_title('Mean Popularity by Keyword Cluster')

plt.tight_layout()
save_figure(fig, 'kw_cluster_profiles')
plt.show()

print("\n=== Cluster Profiles ===")
print(cluster_profile_df[['cluster', 'size', 'top_keywords', 'mean_rating', 'mean_votes']]
      .to_string(index=False))

<a id="11-categories"></a>
## 11. Keyword Category Analysis (Manual Categorization)

While clustering finds statistical patterns, **manual categorization** applies domain knowledge. We assign the top ~100 keywords to interpretable thematic categories to see how different content dimensions relate to ratings, popularity, and temporal trends.

In [ ]:
# Manual categorization of top keywords into thematic categories
keyword_categories = {
    'Relationship': [
        'husband wife relationship', 'father son relationship', 'father daughter relationship',
        'mother son relationship', 'friendship', 'mother daughter relationship',
        'brother sister relationship', 'love', 'marriage', 'infidelity', 'jealousy',
        'boyfriend girlfriend relationship', 'love triangle', 'divorce', 'ex husband ex wife relationship',
        'unrequited love', 'couple'
    ],
    'Violence & Crime': [
        'murder', 'death', 'blood', 'violence', 'gun', 'fight', 'police',
        'revenge', 'kidnapping', 'corpse', 'knife', 'chase', 'torture',
        'prison', 'shooting', 'gangster', 'drug dealer', 'robbery'
    ],
    'Family': [
        'family relationships', 'father', 'mother', 'son', 'daughter', 'child',
        'pregnancy', 'baby', 'brother', 'sister', 'orphan', 'family'
    ],
    'Identity & Coming-of-Age': [
        'coming of age', 'teenage boy', 'teenage girl', 'adolescence', 'gay',
        'lesbian', 'homosexuality', 'sexuality', 'identity', 'transgender',
        'racism', 'immigrant', 'refugee'
    ],
    'Production Type': [
        'based on novel', 'independent film', 'flashback', 'voice over narration',
        'title spoken by character', 'based on true story', 'remake',
        'psychotronic film', 'surrealism', 'black comedy', 'ensemble cast',
        'anthology', 'nonlinear timeline'
    ],
    'Setting & Place': [
        'paris france', 'london england', 'berlin germany', 'new york city',
        'village', 'island', 'countryside', 'apartment', 'hospital', 'school',
        'church', 'train', 'forest', 'beach', 'bar', 'restaurant'
    ],
    'Historical & War': [
        'world war two', 'world war one', 'nazi', 'holocaust', 'soldier',
        'war', 'resistance', 'occupation', 'communism', 'revolution',
        'cold war', 'terrorism'
    ],
    'Sensual & Physical': [
        'female nudity', 'female rear nudity', 'male nudity', 'kiss', 'sex scene',
        'cigarette smoking', 'drinking', 'alcohol', 'drug use', 'nudity',
        'male rear nudity', 'topless female nudity', 'sex'
    ]
}

# For each film, count how many keywords fall into each category
for cat, cat_kws in keyword_categories.items():
    df[f'cat_{cat}'] = df['keyword_list'].apply(
        lambda x: sum(1 for k in (x if isinstance(x, list) else []) if k in cat_kws))

# Count films that have at least one keyword in each category
cat_film_counts = {}
for cat in keyword_categories:
    cat_film_counts[cat] = (df[f'cat_{cat}'] > 0).sum()

fig, ax = plt.subplots(figsize=(10, 5))
cats_sorted = sorted(cat_film_counts.items(), key=lambda x: x[1], reverse=True)
ax.bar([c[0] for c in cats_sorted], [c[1] for c in cats_sorted], color='steelblue')
ax.set_ylabel('Number of Films')
ax.set_title('Films per Keyword Category\n(Films containing at least one keyword in category)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure(fig, 'kw_category_counts')
plt.show()

In [ ]:
# Rating distribution by keyword category
cat_rating_data = []
for cat in keyword_categories:
    mask = df[f'cat_{cat}'] > 0
    for rating in df.loc[mask, 'imdbRating']:
        cat_rating_data.append({'category': cat, 'rating': rating})

cat_rating_df = pd.DataFrame(cat_rating_data)

fig, ax = plt.subplots(figsize=(12, 6))
order = cat_rating_df.groupby('category')['rating'].median().sort_values(ascending=False).index
sns.boxplot(data=cat_rating_df, x='category', y='rating', order=order, ax=ax, palette='muted')
ax.axhline(df['imdbRating'].mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Overall mean: {df["imdbRating"].mean():.2f}')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_ylabel('IMDb Rating')
ax.set_title('Rating Distribution by Keyword Category')
ax.legend()
plt.tight_layout()
save_figure(fig, 'kw_category_ratings')
plt.show()

In [ ]:
# Category prevalence over time
cat_decade = pd.DataFrame()
for cat in keyword_categories:
    prevalence = []
    for decade in valid_decades:
        dec_mask = df['decade'] == decade
        n_dec = dec_mask.sum()
        cat_in_dec = (df.loc[dec_mask, f'cat_{cat}'] > 0).sum()
        prevalence.append(cat_in_dec / n_dec * 100)
    cat_decade[cat] = prevalence
cat_decade.index = valid_decades

fig, ax = plt.subplots(figsize=(14, 7))
cat_decade.plot(ax=ax, marker='o')
ax.set_xlabel('Decade')
ax.set_ylabel('% of Films with Category Keywords')
ax.set_title('Keyword Category Trends Over Time')
ax.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
save_figure(fig, 'kw_category_trends')
plt.show()

<a id="12-rarity"></a>
## 12. Keyword Rarity & Uniqueness

Are films with unusual keyword profiles rated differently from generic ones? This section quantifies **thematic uniqueness** and explores its relationship to quality and visibility.

In [ ]:
# Distribution of keyword rarity
rarity_bins = pd.cut(keyword_counts, bins=[0, 1, 2, 5, 10, 50, keyword_counts.max()],
                     labels=['1 film', '2 films', '3-5', '6-10', '11-50', '50+'])
rarity_dist = rarity_bins.value_counts().reindex(['1 film', '2 films', '3-5', '6-10', '11-50', '50+'])

fig, ax = plt.subplots(figsize=(10, 5))
bars = rarity_dist.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Keyword Frequency (appears in N films)')
ax.set_ylabel('Number of Unique Keywords')
ax.set_title('Keyword Rarity Distribution\nMost keywords are extremely rare')

# Add percentage labels
total = len(keyword_counts)
for i, val in enumerate(rarity_dist):
    ax.text(i, val + 200, f'{val/total*100:.1f}%', ha='center', fontsize=9)

plt.xticks(rotation=0)
plt.tight_layout()
save_figure(fig, 'kw_rarity_distribution')
plt.show()

In [ ]:
# Film uniqueness score: average rarity of a film's keywords
# Higher score = more unusual keyword profile
kw_freq_map = keyword_counts.to_dict()

def uniqueness_score(kw_list):
    if not isinstance(kw_list, list) or len(kw_list) == 0:
        return 0
    return np.mean([1.0 / kw_freq_map.get(kw, 1) for kw in kw_list])

df['uniqueness'] = df['keyword_list'].apply(uniqueness_score)

# Scatter: uniqueness vs rating
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df['uniqueness'], df['imdbRating'], alpha=0.3, s=15, c='steelblue')
ax.set_xlabel('Keyword Uniqueness Score (higher = more unusual themes)')
ax.set_ylabel('IMDb Rating')
ax.set_title('Thematic Uniqueness vs. Rating\nDo films with unusual keyword profiles rate differently?')

# Add correlation
r, p = stats.pearsonr(df['uniqueness'].dropna(), df['imdbRating'].dropna())
ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.3f}', transform=ax.transAxes,
        fontsize=10, verticalalignment='top')

plt.tight_layout()
save_figure(fig, 'kw_uniqueness_vs_rating')
plt.show()

In [ ]:
# Most unique and most generic films
print("=== Top 10 Most Thematically Unique Films ===")
print(df.nlargest(10, 'uniqueness')[['originalTitle', 'releaseYear', 'imdbRating',
                                      'keyword_count', 'uniqueness', 'genres']]
      .to_string(index=False))

print("\n=== Top 10 Most Generically-Keyworded Films ===")
# Only films with a reasonable number of keywords
generic = df[df['keyword_count'] >= 20].nsmallest(10, 'uniqueness')
print(generic[['originalTitle', 'releaseYear', 'imdbRating',
               'keyword_count', 'uniqueness', 'genres']]
      .to_string(index=False))

# Rare keyword spotlight
print("\n=== Keywords Appearing in Exactly 1 Film (sample of 20) ===")
hapax_kws = keyword_counts[keyword_counts == 1].index.tolist()
np.random.seed(42)
sample_hapax = np.random.choice(hapax_kws, size=min(20, len(hapax_kws)), replace=False)
for kw in sample_hapax:
    film = df[df['keywords'].str.contains(rf'(?:^|, ){kw}(?:,|$)', na=False, regex=True)]
    if len(film) > 0:
        title = film.iloc[0]['originalTitle']
        print(f"  \"{kw}\" → {title}")

<a id="13-runtime"></a>
## 13. Keywords & Runtime

Do certain themes correlate with longer or shorter films? And do films with more keywords tend to be longer (more content to tag)?

In [ ]:
# Mean runtime by keyword category
cat_runtime = {}
for cat in keyword_categories:
    mask = df[f'cat_{cat}'] > 0
    cat_runtime[cat] = df.loc[mask, 'runtimeMinutes'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
sorted_cats = sorted(cat_runtime.items(), key=lambda x: x[1], reverse=True)
ax.barh([c[0] for c in sorted_cats], [c[1] for c in sorted_cats], color='steelblue')
ax.axvline(df['runtimeMinutes'].mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Overall mean: {df["runtimeMinutes"].mean():.0f} min')
ax.set_xlabel('Mean Runtime (minutes)')
ax.set_title('Mean Runtime by Keyword Category')
ax.legend()
plt.tight_layout()
save_figure(fig, 'kw_runtime_by_category')
plt.show()

In [ ]:
# Correlation: keyword count vs. runtime
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['keyword_count'], df['runtimeMinutes'], alpha=0.3, s=15, c='steelblue')
ax.set_xlabel('Number of Keywords')
ax.set_ylabel('Runtime (minutes)')
ax.set_title('Keyword Count vs. Runtime\nDo films with more keywords tend to be longer?')

# Add regression line
mask = df['keyword_count'].notna() & df['runtimeMinutes'].notna()
slope, intercept, r_val, p_val, _ = stats.linregress(
    df.loc[mask, 'keyword_count'], df.loc[mask, 'runtimeMinutes'])
x_line = np.linspace(df['keyword_count'].min(), df['keyword_count'].max(), 100)
ax.plot(x_line, intercept + slope * x_line, 'r--', alpha=0.7,
        label=f'r = {r_val:.3f}, p = {p_val:.1e}')
ax.legend()
plt.tight_layout()
save_figure(fig, 'kw_count_vs_runtime')
plt.show()

<a id="14-coproductions"></a>
## 14. Keywords & Co-productions

Do international co-productions have different thematic profiles than single-country productions? Are co-productions more "international" in their keyword signatures?

In [ ]:
# Keywords over/under-represented in co-productions
coprod = df[df['is_coproduction']]
single = df[~df['is_coproduction']]

coprod_kw = split_field(coprod['keywords']).value_counts()
single_kw = split_field(single['keywords']).value_counts()

coprod_pct = coprod_kw / len(coprod)
single_pct = single_kw / len(single)

common_cp = set(coprod_kw[coprod_kw >= 10].index) & set(single_kw[single_kw >= 10].index)
cp_ratios = []
for kw in common_cp:
    ratio = coprod_pct.get(kw, 0) / max(single_pct.get(kw, 0.001), 0.001)
    cp_ratios.append({'keyword': kw, 'ratio': ratio})

cp_df = pd.DataFrame(cp_ratios).sort_values('ratio', ascending=False)

top_coprod = cp_df.head(15)
top_single = cp_df.tail(15).iloc[::-1]
combined_cp = pd.concat([top_coprod, top_single])

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#27ae60' if r > 1 else '#8e44ad' for r in combined_cp['ratio']]
ax.barh(range(len(combined_cp)), np.log2(combined_cp['ratio']), color=colors)
ax.set_yticks(range(len(combined_cp)))
ax.set_yticklabels(combined_cp['keyword'])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('log₂(Co-prod / Single ratio)  ←Single-country | Co-production→')
ax.set_title('Keywords: Co-productions vs. Single-Country Films')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'kw_coprod_vs_single')
plt.show()

In [ ]:
# Keyword diversity: do co-productions have richer keyword profiles?
coprod_kw_count = coprod['keyword_count'].mean()
single_kw_count = single['keyword_count'].mean()

t_stat, p_val = stats.ttest_ind(coprod['keyword_count'], single['keyword_count'])

print("=== Keyword Diversity: Co-productions vs. Single-Country ===")
print(f"\nMean keywords per film:")
print(f"  Co-productions:  {coprod_kw_count:.1f} (n={len(coprod)})")
print(f"  Single-country:  {single_kw_count:.1f} (n={len(single)})")
print(f"\nt-test: t={t_stat:.2f}, p={p_val:.4f}")
print(f"{'Significant' if p_val < 0.05 else 'Not significant'} at α=0.05")

<a id="15-predictive"></a>
## 15. Predictive: Can Keywords Predict Rating Tier?

This section tests whether keywords contain enough signal to predict a film's quality tier. This is **correlational, not causal** — keywords don't cause ratings, but they may encode information about the type of film being made.

In [ ]:
# Define rating tiers
df['rating_tier'] = pd.cut(df['imdbRating'], bins=[0, 5.5, 7.0, 10],
                           labels=['Low (<5.5)', 'Medium (5.5-7.0)', 'High (>7.0)'])
print("Rating tier distribution:")
print(df['rating_tier'].value_counts().sort_index())

# Use the keyword matrix built earlier (frequent_kws, from Section 6)
print(f"\nFeature matrix: {kw_matrix.shape}")

In [ ]:
# Train RandomForest to predict rating tier
y = df['rating_tier'].astype(str)
X = kw_matrix

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
scores = cross_val_score(clf, X, y, cv=5, scoring='f1_macro')
print(f"5-fold CV Macro-F1: {scores.mean():.3f} (±{scores.std():.3f})")

accuracy = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f"5-fold CV Accuracy: {accuracy.mean():.3f} (±{accuracy.std():.3f})")

# Fit on full data for feature importance and confusion matrix
clf.fit(X, y)
y_pred = clf.predict(X)  # Training set (for illustration — CV scores above are honest)

print("\n=== Classification Report (full training set — see CV scores for honest estimate) ===")
print(classification_report(y, y_pred))

In [ ]:
# Feature importance: which keywords matter most for prediction?
importances = clf.feature_importances_
top_20_idx = importances.argsort()[-20:][::-1]
top_20_features = [(frequent_kws[i], importances[i]) for i in top_20_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), [f[1] for f in top_20_features], color='steelblue')
ax.set_yticks(range(20))
ax.set_yticklabels([f[0] for f in top_20_features])
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 20 Most Predictive Keywords for Rating Tier')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'kw_prediction_feature_importance')
plt.show()

In [ ]:
# Confusion matrix
labels = ['Low (<5.5)', 'Medium (5.5-7.0)', 'High (>7.0)']
cm = confusion_matrix(y, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels,
            yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix: Keyword-Based Rating Tier Prediction\n(Training set — see CV scores for honest estimate)')
plt.tight_layout()
save_figure(fig, 'kw_prediction_confusion_matrix')
plt.show()

<a id="16-summary"></a>
## 16. Summary & Key Takeaways

### Key Findings

*(Fill in after running all sections. Structure around actionable insights:)*

1. **Keyword distribution** — Keywords follow a power-law distribution (Zipf's law). The vast majority of keywords are rare, while a small number dominate.

2. **Rating signals** — Certain keywords significantly correlate with higher/lower ratings. "Prestige" markers (based on novel, etc.) tend to rate higher; genre-specific content markers may rate lower.

3. **Mainstream vs. arthouse** — Clear keyword signatures distinguish popular from niche films, providing a vocabulary for positioning decisions.

4. **Genre encoding** — Keywords contain substantial genre information (see prediction F1 scores), but also carry orthogonal thematic detail that genres alone miss.

5. **Cultural differences** — National cinemas have distinctive keyword profiles. Nordic cinema has its own thematic signature.

6. **Temporal shifts** — Some themes are rising, others declining — reflecting changing cultural preoccupations in European cinema.

7. **Thematic clusters** — Keywords naturally cluster into interpretable themes that correlate with different rating and popularity profiles.

8. **Predictive power** — Keywords have moderate predictive power for rating tier, suggesting they encode meaningful information about film quality/reception.

---

### Limitations

| Limitation | Impact |
|-----------|--------|
| Keywords are user-contributed, non-standardized | Synonyms may be split across entries; keyword coverage varies |
| IMDb user base bias (male, English-speaking, younger) | Keyword tagging and ratings reflect this demographic |
| Correlation ≠ causation | Keywords don't cause ratings — they describe content that happens to rate higher or lower |
| Sample size (2,000 films) | Some keyword-level analyses may have small n; rare keywords are noisy |
| No weighting by keyword importance | All keywords treated equally, but some are more descriptive than others |

---

### Connection to Publikum's methodology

This keyword analysis is a simplified analog of Publikum's **PlotBounce** approach, which uses 500 "expectation clusters" built from 163,000 films. Our analysis demonstrates that even with standard IMDb keywords, meaningful thematic patterns emerge that can inform film positioning — providing a foundation for the **References** pillar of Publikum's three-pillar methodology.

> *"Keywords are not the story, but they are the shadow the story casts onto a database."*